<div style="display: flex; background-color: RGB(12,38,153);" >
<h1 style="margin: auto; padding: 30px; ">APPLICATION DE DETECTION AUTOMATIQUE DES FAUX BILLETS</h1>
</div>

In [1]:
import pandas as pd                                                                                                       
import joblib

df_test = pd.read_csv("billets_production.csv", sep = ",")                                          ## vérifier que le fichier est bien dans le même dossier que votre script. Si besoin, modifier le nom du fichier ici.

colonnes_test = ["height_left", "height_right", "margin_low", "margin_up", "length"]                          
ids = df_test["id"]
df_test = df_test[colonnes_test]                                                                                          


nb_val_manquantes = df_test.isna().sum().sum()                                                                            
if nb_val_manquantes > 0 :                                                                                                
    print(f"Attention : {nb_val_manquantes} donnée(s) manquante(s) à corriger dans le tableau.")                          
else :                                                                                                                    
    pipeline = joblib.load("pipeline_scaled.joblib")                                                ## vérifier que le fichier est bien dans le même dossier que votre script.
    pred_test = pipeline.predict(df_test)
    proba_test = pipeline.predict_proba(df_test)
    df_test["%proba_faux"] = (proba_test[:, 0] * 100).round(2)
    df_test["%proba_vrai"] = (proba_test[:, 1] * 100).round(2)
    df_test["is_genuine"] = pred_test
    df_test["id"] = ids
    vrais_billets = df_test.loc[df_test["is_genuine"] == True, "id"].tolist()      
    faux_billets = df_test.loc[df_test["is_genuine"] == False, "id"].tolist()
    print(f"Vrais billets = {len(vrais_billets)}  -> id : {vrais_billets}")   
    print(f"Faux billets = {len(faux_billets)}   -> id : {faux_billets}\n")
    print(f"Vérification des probabilités de prévision pour confirmer les résultats :")
    print(df_test[["id", "is_genuine", "%proba_vrai", "%proba_faux"]])


Vrais billets = 2  -> id : ['A_4', 'A_5']
Faux billets = 3   -> id : ['A_1', 'A_2', 'A_3']

Vérification des probabilités de prévision pour confirmer les résultats :
    id  is_genuine  %proba_vrai  %proba_faux
0  A_1       False         0.13        99.87
1  A_2       False         0.02        99.98
2  A_3       False         0.02        99.98
3  A_4        True        94.03         5.97
4  A_5        True        99.97         0.03
